# Computer Vision Assignment — PPE (Helmet & Safety Vest) Detection

**Course:** SMIT0038-AIDS-ZA-2026 TT 8 to 11
**Deadline:** 28th September

## 1. Idea & Problem Statement

**Chosen problem:** Automatic detection of Personal Protective Equipment (PPE) — specifically **helmets** and **safety vests** — on construction/industrial site footage.

**Why this idea:** Manual monitoring of workers for safety-gear compliance on a busy site is slow and error-prone. An automated detector can flag workers without a helmet or vest in real time from CCTV/video, improving site safety and giving a measurable compliance rate.

**Classes to detect:**
- `helmet`
- `no_helmet`
- `vest`
- `no_vest`

> Edit this cell with your own wording/idea if you want to change the exact framing — the pipeline below works for any single-object or multi-class detection idea, just swap the class names and images.

## 2. Data Collection & Annotation

**What you need to do yourself (this cannot be automated — it's the "unique dataset" part of the assignment):**

1. Collect 150–300 images/frames of workers with and without helmets/vests. Sources:
   - Your own phone photos/videos on any site you can access, OR
   - Frame-extract from royalty-free construction-site videos (Pexels/Pixabay — free to use), OR
   - A mix of both, cropped/resized to remove any identifying background if privacy is a concern.
2. Upload all images to a new [Roboflow](https://roboflow.com) project (free account).
3. Annotate every image by drawing a bounding box + label for each class above.
4. In Roboflow: **Generate** a version → apply an 80/10/10 train/val/test split → **Export** in **YOLOv8** format → copy the generated download code snippet (it contains your private API key).
5. Paste your snippet's `workspace`, `project`, and `version` values into the cell below.

The frame-extraction helper below saves you time if you're pulling images from a video instead of individual photos.

In [ ]:
# OPTIONAL helper: extract frames from a video you recorded/downloaded, one every N frames,
# so you have a folder of images ready to upload to Roboflow for annotation.

import cv2, os

VIDEO_PATH = "/content/site_video.mp4"   # <-- upload your video and set this path
OUTPUT_DIR = "/content/extracted_frames"
FRAME_SKIP = 15                           # save 1 frame every 15 -> ~2 frames/sec for a 30fps video

os.makedirs(OUTPUT_DIR, exist_ok=True)

if os.path.exists(VIDEO_PATH):
    cap = cv2.VideoCapture(VIDEO_PATH)
    count, saved = 0, 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if count % FRAME_SKIP == 0:
            cv2.imwrite(f"{OUTPUT_DIR}/frame_{saved:04d}.jpg", frame)
            saved += 1
        count += 1
    cap.release()
    print(f"Saved {saved} frames to {OUTPUT_DIR}")
else:
    print("No video found at VIDEO_PATH — skip this cell if you're uploading photos directly.")

In [ ]:
# Install required packages
!pip install -q ultralytics roboflow

In [ ]:
# Download YOUR annotated dataset from Roboflow (YOLOv8 format)
# Replace the values below with the ones from YOUR Roboflow export snippet.

from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
project = rf.workspace("YOUR_WORKSPACE").project("YOUR_PROJECT_NAME")
dataset = project.version(1).download("yolov8")

DATA_YAML = f"{dataset.location}/data.yaml"
print("Dataset ready at:", dataset.location)
print("data.yaml path:", DATA_YAML)

## 3. Verify the Dataset
Quick sanity check: confirm class names, image counts, and preview a few annotated samples before training.

In [ ]:
import yaml, glob, random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

with open(DATA_YAML) as f:
    data_cfg = yaml.safe_load(f)

print("Classes:", data_cfg["names"])

train_images = glob.glob(f"{dataset.location}/train/images/*")
print(f"Train images: {len(train_images)}")

def show_sample(img_path):
    label_path = img_path.replace("images", "labels").rsplit(".", 1)[0] + ".txt"
    img = Image.open(img_path)
    w, h = img.size
    fig, ax = plt.subplots(1)
    ax.imshow(img)
    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                cls, x, y, bw, bh = map(float, line.split())
                x1 = (x - bw / 2) * w
                y1 = (y - bh / 2) * h
                rect = patches.Rectangle((x1, y1), bw * w, bh * h, linewidth=2,
                                          edgecolor="lime", facecolor="none")
                ax.add_patch(rect)
                ax.text(x1, y1 - 5, data_cfg["names"][int(cls)], color="lime", fontsize=9)
    plt.axis("off")
    plt.show()

for p in random.sample(train_images, min(3, len(train_images))):
    show_sample(p)

## 4. Model Training (YOLOv8)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")   # nano model — fast to train, good for a course assignment

results = model.train(
    data=DATA_YAML,
    epochs=60,
    imgsz=640,
    batch=16,
    patience=15,
    project="ppe_detection",
    name="yolov8_ppe"
)

## 5. Evaluation & Results
Ultralytics auto-saves confusion matrix, PR curve, F1 curve, and loss/mAP graphs to the run folder — this cell displays them, plus prints the final metrics.

In [ ]:
metrics = model.val()

print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

In [ ]:
run_dir = f"ppe_detection/yolov8_ppe"

for fname in ["confusion_matrix.png", "results.png", "PR_curve.png", "F1_curve.png"]:
    path = f"{run_dir}/{fname}"
    if os.path.exists(path):
        img = Image.open(path)
        plt.figure(figsize=(8, 6))
        plt.imshow(img)
        plt.axis("off")
        plt.title(fname)
        plt.show()

## 6. Inference on Test Images
Run the trained model on the held-out test split and display the detections — this is the "final outputs / test images result" the assignment asks for.

In [ ]:
best_weights = f"{run_dir}/weights/best.pt"
trained_model = YOLO(best_weights)

test_images = glob.glob(f"{dataset.location}/test/images/*")
sample_tests = random.sample(test_images, min(5, len(test_images)))

test_results = trained_model.predict(sample_tests, conf=0.4, save=True)

for r in test_results:
    plotted = r.plot()  # returns a BGR numpy image with boxes drawn
    plt.figure(figsize=(8, 6))
    plt.imshow(plotted[..., ::-1])  # BGR -> RGB
    plt.axis("off")
    plt.show()

## 7. Conclusion (fill in after running)

- **Final mAP50:** _fill in from Section 5 output_
- **Final Precision / Recall:** _fill in from Section 5 output_
- **Observations:** _e.g. which class was hardest to detect, effect of dataset size, ideas for improvement (more images, more epochs, larger YOLO model, etc.)_

## 8. Deliverables Checklist
- [ ] This notebook, fully run with outputs visible (download as .ipynb)
- [ ] Roboflow dataset link (make sure sharing is enabled)
- [ ] Short report/presentation summarizing idea, dataset, training graphs, and output predictions (see accompanying report template)